# 5.3.1 · Auditoria das repetições de id_gleba2
## Evidências para responder às perguntas do orientador

**Pronto para o Colab:** os caminhos e o ID da Base04 com `ID_GLEBA2` estão preenchidos. Execute em ordem. O notebook lê a base sem modificá-la e gera tabelas, casos para conferência e um texto de resultados.

### Perguntas respondidas
1. Quantas representações geométricas se repetem? Quantos registros adicionais isso representa?
2. Há repetição no **mesmo ano**? Em **anos diferentes**?
3. Uma geometria está associada a **operações distintas** (`REF_BACEN`)? E a ordens distintas?
4. Há repetição da geometria dentro da **mesma operação e ordem**?
5. Existem linhas iguais em todas as colunas? E linhas de conteúdo igual quando desconsiderados apenas os identificadores artificiais?
6. É possível afirmar que foi **o mesmo produtor**? O que permanece sem comprovação?

### Um ajuste necessário na redação do método
O [Base05 consultado](https://colab.research.google.com/drive/1xpHJ0rE1mPcCap-72Ifu00fBW44zuMae) cria uma assinatura SHA-256 do **texto WKT exato**, sem normalizar vértices, ordem dos anéis, precisão ou espaços. Cada grupo recebe o **menor ID_GLEBA** de suas linhas. Portanto, este diagnóstico trata de **repetições da mesma representação WKT reconhecidas pelo ID_GLEBA2**. Geometrias espacialmente equivalentes, mas escritas de formas diferentes, podem continuar com IDs diferentes. Não descreva o procedimento como padronização geométrica ou teste geral de igualdade topológica.

### Cuidados de interpretação
- A unidade é o **registro territorial**, preservando repetições de operações e ordens.
- `NU_ORDEM_GLEBA` identifica a ordem **em conjunto com REF_BACEN**; o número isolado não é uma chave global.
- O ano padrão é `ANO_BASE`, ou seja, **ano do arquivo territorial**. As conclusões dirão “ano de origem”; não o confundir automaticamente com ano de contratação.
- Operações diferentes na mesma representação territorial não provam que seja o mesmo produtor e não comprovam solicitação, aprovação ou desembolso.
- Repetição de geometria não é, por si só, erro. Linhas idênticas e repetições dentro da mesma chave são **ocorrências para conferência**, sem exclusão automática.
- Também são consideradas representações de pontos. Não se chama toda linha da base de polígono.


## 1. Bibliotecas e acesso ao Drive

In [ ]:
%pip -q install "pandas>=2.2,<3" "numpy>=1.26,<3" "google-api-python-client>=2,<3" "google-auth>=2,<3"
from google.colab import drive
drive.mount("/content/drive")


ValueError: mount failed

## 2. Configuração já preenchida
Não é necessário alterar os caminhos se a estrutura do seu Drive for a mesma. Se o arquivo não for encontrado no caminho, há download autenticado pelo ID confirmado.

A coluna `ID_GLEBA2` é localizada sem diferenciar maiúsculas de minúsculas. **A ordem usada é NU_ORDEM_GLEBA**, conforme o dicionário da Base04. Os códigos permanecem textuais; zeros à esquerda não são removidos.

`COLUNA_BENEFICIARIO = None` é o padrão, pois o dicionário consultado não documenta um identificador individual do beneficiário. Porte, programa e categoria do emitente **não identificam a pessoa**. Preencha esse parâmetro somente se existir na base uma coluna apropriada e documentada. Nos relatórios, os valores dessa coluna serão substituídos por hashes, mas continuam sendo identificadores pseudonimizados, não anonimizados.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
PASTA_RELATORIOS = Path("/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios")
ARQUIVO_BASE = PASTA_RELATORIOS / "Base04_enriquecimento_detalhado" / "sicor_cerrado_2021_2025_enriquecido_detalhado_com_id_gleba2.csv"
DRIVE_ID_BASE = "1AdXP97WCkiecAbvIFTKbFWU7WyOSzjTF"
COLUNAS = {
    "id_gleba": "ID_GLEBA",
    "id_gleba2": "ID_GLEBA2",
    "ref_bacen": "REF_BACEN",
    "ordem": "NU_ORDEM_GLEBA",
    "ano": "ANO_BASE",
    "wkt": "GT_GEOMETRIA",
}
COLUNA_BENEFICIARIO = None
COLUNA_INDICE = "NU_INDICE"    # informativa, quando presente
ANOS = [2021, 2022, 2023, 2024, 2025]
TAMANHO_BLOCO = 20_000
SEPARADOR = ";"
CODIFICACAO = "utf-8-sig"
BAIXAR_ZIP = True
PERMITIR_DOWNLOAD_POR_ID = True
# Referências históricas, apenas para comparação; não são usadas para preencher resultados.
TOTAIS_ANTERIORES = {"registros":480679, "id_gleba2":395575}
EXECUCAO = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
RAIZ_SAIDA = PASTA_RELATORIOS if PASTA_RELATORIOS.is_dir() else Path("/content")
SAIDA = RAIZ_SAIDA / "auditoria_repeticoes_5_3_1" / EXECUCAO
SAIDA.mkdir(parents=True, exist_ok=True)
print("Entrada:", ARQUIVO_BASE)
print("Saída:", SAIDA)


## 3. Localizar a base e conferir o cabeçalho

In [ ]:
import csv, hashlib, json, zipfile, itertools, platform, gc
import numpy as np
import pandas as pd
from IPython.display import display, HTML

def localizar_base():
    if ARQUIVO_BASE.is_file():
        st=ARQUIVO_BASE.stat()
        return ARQUIVO_BASE, {"acesso":"caminho_montado", "tamanho_bytes":st.st_size, "mtime":st.st_mtime}
    if not PERMITIR_DOWNLOAD_POR_ID:
        raise FileNotFoundError(f"Arquivo não encontrado: {ARQUIVO_BASE}")
    from google.colab import auth
    import google.auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload
    auth.authenticate_user()
    credenciais,_=google.auth.default()
    servico=build("drive","v3",credentials=credenciais,cache_discovery=False)
    meta=servico.files().get(fileId=DRIVE_ID_BASE,fields="id,name,size,modifiedTime,md5Checksum",supportsAllDrives=True).execute()
    if meta["name"]!=ARQUIVO_BASE.name:
        raise ValueError("O arquivo mudou de nome. Confira o ID antes de continuar: "+meta["name"])
    destino=Path("/content")/ARQUIVO_BASE.name
    marcador=destino.with_suffix(".origem.json")
    pronto=destino.exists() and marcador.exists() and json.loads(marcador.read_text())==meta and destino.stat().st_size==int(meta["size"])
    if not pronto:
        tmp=destino.with_suffix(".part")
        req=servico.files().get_media(fileId=DRIVE_ID_BASE,supportsAllDrives=True)
        with tmp.open("wb") as arq:
            dl=MediaIoBaseDownload(arq,req,chunksize=32*1024*1024);fim=False
            while not fim:
                status,fim=dl.next_chunk(num_retries=3)
                if status: print(f"Download: {100*status.progress():.0f}%")
        if tmp.stat().st_size!=int(meta["size"]):raise RuntimeError("Download incompleto.")
        tmp.replace(destino);marcador.write_text(json.dumps(meta),encoding="utf-8")
    return destino, {"acesso":"download_autenticado",**meta}

def achar_coluna(colunas,nome,obrigatoria=True):
    if nome in colunas:return nome
    candidatos=[x for x in colunas if x.lstrip("#").casefold()==nome.lstrip("#").casefold()]
    if len(candidatos)==1:return candidatos[0]
    if not candidatos and not obrigatoria:return None
    raise ValueError(f"Não foi possível resolver {nome!r} sem ambiguidade. Colunas: {colunas}")

def conferir_cabecalho(p):
    with p.open("r",encoding=CODIFICACAO,newline="") as arq:
        cab=next(csv.reader(arq,delimiter=SEPARADOR))
    if len(cab)!=len(set(cab)):raise ValueError("Cabeçalhos repetidos no arquivo de entrada.")
    mapa={k:achar_coluna(cab,v) for k,v in COLUNAS.items()}
    return cab,mapa

CAMINHO, META_ENTRADA = localizar_base()
CABECALHO, MAPA = conferir_cabecalho(CAMINHO)
BENEFICIARIO = achar_coluna(CABECALHO,COLUNA_BENEFICIARIO) if COLUNA_BENEFICIARIO else None
INDICE = achar_coluna(CABECALHO,COLUNA_INDICE,False) if COLUNA_INDICE else None
COLUNAS_SEM_IDS = [x for x in CABECALHO if x not in {MAPA["id_gleba"],MAPA["id_gleba2"]}]
print("Colunas utilizadas:", MAPA)
print("Identificador de beneficiário:", BENEFICIARIO or "não informado; identidade do produtor não será inferida")
print("Na comparação de conteúdo, excluem-se SOMENTE:",MAPA["id_gleba"],"e",MAPA["id_gleba2"])


## 4. Ler em blocos e preparar assinaturas
São retidas em memória apenas as chaves necessárias e assinaturas SHA-256. Os atributos completos são lidos para comparar o conteúdo das linhas, mas não são todos copiados para os relatórios.

Há três comparações diferentes:
- Texto WKT exatamente igual: reproduz o critério do Base05.
- Todas as colunas iguais: inclui `ID_GLEBA` e `ID_GLEBA2`.
- Conteúdo igual sem os dois IDs artificiais: mantém todas as demais colunas, inclusive ano, índice, valores e atributos administrativos.

As assinaturas de conteúdo usam uma lista JSON ordenada e sem ambiguidades de separador. Os candidatos a duplicação são conferidos por igualdade dos valores lidos na seção 7. Espaços são preservados nessa comparação e no WKT; apenas as chaves usadas nos agrupamentos têm espaços externos removidos. Nenhum número de identificação é convertido automaticamente de texto para número nos agrupamentos.


In [ ]:
MARCADORES_AUSENTES = {"", "null", "none", "nan", "<na>"}
def limpar_chave(s):
    x=s.astype("string").fillna("").str.strip()
    return x.mask(x.str.casefold().isin(MARCADORES_AUSENTES),pd.NA)

def assinatura_valores(valores):
    payload=json.dumps(list(valores),ensure_ascii=False,separators=(",",":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def preparar_bloco(bloco,inicio):
    bloco=bloco.fillna("").reset_index(drop=True)
    d=pd.DataFrame({"linha_arquivo":np.arange(inicio+1,inicio+len(bloco)+1,dtype=np.int64)})
    for k in ["id_gleba","id_gleba2","ref_bacen","ordem"]:d[k]=limpar_chave(bloco[MAPA[k]])
    ano_txt=limpar_chave(bloco[MAPA["ano"]])
    ano_num=pd.to_numeric(ano_txt,errors="coerce")
    d["ano"]=ano_num.where(ano_num.notna() & np.isfinite(ano_num) & ano_num.mod(1).eq(0) & ano_num.between(1,9999)).astype("Int64")
    d["ano_original"]=ano_txt
    if INDICE:d["nu_indice"]=limpar_chave(bloco[INDICE])
    w=bloco[MAPA["wkt"]].astype(str)
    ausente=w.str.strip().str.casefold().isin(MARCADORES_AUSENTES)
    d["wkt_sha256"]=[pd.NA if a else hashlib.sha256(t.encode("utf-8")).hexdigest() for t,a in zip(w,ausente)]
    # O tipo é informativo e extraído do texto. Esta etapa não revalida topologia.
    d["tipo_wkt"]=w.str.extract(r"^\s*(?:SRID=\d+;)?([A-Za-z]+)",expand=False).str.upper()
    d["linha_sha256"]=[assinatura_valores(v) for v in bloco[CABECALHO].itertuples(index=False,name=None)]
    d["conteudo_sha256"]=[assinatura_valores(v) for v in bloco[COLUNAS_SEM_IDS].itertuples(index=False,name=None)]
    if BENEFICIARIO:
        b=limpar_chave(bloco[BENEFICIARIO])
        d["beneficiario_hash"]=[pd.NA if pd.isna(v) else hashlib.sha256(v.encode("utf-8")).hexdigest() for v in b]
    return d

partes=[];n_lidas=0
for i,bloco in enumerate(pd.read_csv(CAMINHO,sep=SEPARADOR,encoding=CODIFICACAO,dtype=str,keep_default_na=False,
                                    chunksize=TAMANHO_BLOCO,on_bad_lines="error",skip_blank_lines=False),1):
    partes.append(preparar_bloco(bloco,n_lidas));n_lidas+=len(bloco)
    if i==1 or i%5==0:print(f"{n_lidas:,} registros lidos")
    del bloco
base=pd.concat(partes,ignore_index=True);del partes;gc.collect()
print("Total lido:",len(base))


## 5. Conferir integridade do identificador e cobertura
Verifica se um `id_gleba2` agrupa mais de um WKT, se um mesmo WKT recebeu mais de um `id_gleba2` e se o ID escolhido é o menor `ID_GLEBA` do grupo, conforme o Base05. Falhas na correspondência ID ↔ WKT impedem interpretar os grupos como repetições da mesma representação.

Campos essenciais ausentes são exportados e interrompem a auditoria para evitar uma conclusão com chaves incompletas. Anos fora de 2021–2025 são registrados separadamente. Duplicações de `ID_GLEBA` são reportadas, sem eliminar linhas.


In [ ]:
def csv_saida(df,nome):
    df.to_csv(SAIDA/nome,index=False,sep=";",decimal=",",encoding="utf-8-sig",float_format="%.9f")

obrigatorias=["id_gleba","id_gleba2","ref_bacen","ordem","ano","wkt_sha256"]
problemas=base.loc[base[obrigatorias].isna().any(axis=1)].copy()
csv_saida(problemas,"chaves_ou_wkt_ausentes.csv")
if len(problemas):
    raise ValueError(f"{len(problemas)} registros com chaves/ano/WKT ausentes. Consulte chaves_ou_wkt_ausentes.csv antes de prosseguir.")
fora=base.loc[~base.ano.isin(ANOS)].copy();csv_saida(fora,"registros_fora_periodo.csv")
df=base.loc[base.ano.isin(ANOS)].copy()
if df.empty:raise ValueError("Sem registros de 2021 a 2025.")
df["ano"]=df.ano.astype(int)
# Usa o arquivo inteiro para verificar a atribuição canônica, antes de restringir o período.
n_wkt=base.groupby("id_gleba2",sort=False).wkt_sha256.nunique()
n_ids=base.groupby("wkt_sha256",sort=False).id_gleba2.nunique()
conflitos_id=n_wkt[n_wkt>1].rename("wkt_distintos").reset_index()
conflitos_wkt=n_ids[n_ids>1].rename("ids_distintos").reset_index()
csv_saida(conflitos_id,"id_gleba2_com_wkt_diferentes.csv")
csv_saida(conflitos_wkt,"wkt_com_id_gleba2_diferentes.csv")
if len(conflitos_id) or len(conflitos_wkt):
    raise ValueError("A correspondência entre ID_GLEBA2 e WKT não é unívoca. Confira os CSVs de integridade antes de interpretar as repetições.")

orig=pd.to_numeric(base.id_gleba,errors="coerce")
novo=pd.to_numeric(base.id_gleba2,errors="coerce")
inteiros=orig.notna() & novo.notna() & np.isfinite(orig) & np.isfinite(novo) & orig.mod(1).eq(0) & novo.mod(1).eq(0)
if inteiros.all():
    esperado=orig.groupby(base.id_gleba2).transform("min")
    divergencias=base.loc[novo.ne(esperado)].copy()
    verificacao_menor_id="verificada"
else:
    divergencias=base.iloc[0:0].copy();verificacao_menor_id="não verificada: IDs não inteiros"
csv_saida(divergencias,"divergencias_menor_id_gleba.csv")
ids_orig_repetidos=base.loc[base.id_gleba.duplicated(keep=False)].copy()
csv_saida(ids_orig_repetidos,"registros_com_id_gleba_repetido.csv")
controle=pd.DataFrame([
    {"indicador":"registros_lidos","resultado":len(base)},
    {"indicador":"registros_no_periodo","resultado":len(df)},
    {"indicador":"registros_fora_periodo","resultado":len(fora)},
    {"indicador":"id_gleba2_distintos_no_periodo","resultado":df.id_gleba2.nunique()},
    {"indicador":"id_gleba_distintos_repetidos","resultado":ids_orig_repetidos.id_gleba.nunique()},
    {"indicador":"divergencias_na_regra_do_menor_id","resultado":len(divergencias) if verificacao_menor_id=="verificada" else "não verificada"},
])
csv_saida(controle,"controle_integridade.csv");display(controle)
if len(divergencias):print("Atenção: o agrupamento por WKT é unívoco, mas há divergências da regra do menor ID; confira o relatório.")
if len(df)!=TOTAIS_ANTERIORES["registros"] or df.id_gleba2.nunique()!=TOTAIS_ANTERIORES["id_gleba2"]:
    print("Os totais diferem dos resultados anteriores. Os relatórios usarão os valores efetivamente lidos nesta execução.")
print("Referência anual:", MAPA["ano"],"— ano de origem do arquivo territorial.")


## 6. Repetições temporais e relações com operações e ordens
A análise usa contagens de combinações distintas em tabelas auxiliares; a base de entrada permanece intacta.

**Não confunda:** número de geometrias repetidas; número de registros em grupos repetidos; e ocorrências adicionais à primeira. Por exemplo, uma geometria em quatro linhas representa **1 geometria repetida, 4 registros envolvidos e 3 ocorrências adicionais**.

As quatro categorias temporais abaixo são exclusivas. Já os indicadores de repetição em algum ano e repetição entre anos podem se sobrepor. A matriz anual conta IDs compartilhados por cada par de anos e não deve ser somada como um total de geometrias distintas.


In [ ]:
def analisar_repeticoes(d):
    d=d.copy()
    # Factorização de tuplas evita colisões por concatenação de códigos.
    d["ordem_codigo"]=pd.factorize(pd.MultiIndex.from_frame(d[["ref_bacen","ordem"]]),sort=False)[0]
    g=d.groupby("id_gleba2",sort=False).agg(
        registros=("linha_arquivo","size"),anos=("ano","nunique"),operacoes=("ref_bacen","nunique"),
        ordens=("ordem_codigo","nunique"),primeiro_ano=("ano","min"),ultimo_ano=("ano","max"),tipo_wkt=("tipo_wkt","first"))
    gy=d.groupby(["id_gleba2","ano"],sort=False).agg(
        registros=("linha_arquivo","size"),operacoes=("ref_bacen","nunique"),ordens=("ordem_codigo","nunique")).reset_index()
    rep_ano=gy.loc[gy.registros>1].copy()
    ids_intra=set(rep_ano.id_gleba2)
    g["repete_dentro_de_algum_ano"]=g.index.isin(ids_intra)
    g["categoria_temporal"]=np.select(
        [g.registros.eq(1),g.anos.eq(1),g.anos.gt(1)&~g.repete_dentro_de_algum_ano],
        ["Uma ocorrência","Repetição somente dentro de um ano","Repetição entre anos sem repetição interna"],
        default="Repetição entre anos e dentro de algum ano")
    g["categoria_administrativa"]=np.select(
        [g.registros.eq(1),g.operacoes.gt(1),g.ordens.gt(1)],
        ["Uma ocorrência","Repetição com operações distintas","Mesma operação com ordens distintas"],
        default="Mesma operação e mesma ordem")
    anual=[]
    for ano in ANOS:
        a=gy.loc[gy.ano.eq(ano)];rep=a.loc[a.registros>1]
        total=int(a.registros.sum());ng=len(a)
        anual.append({"ano":ano,"registros":total,"id_gleba2_distintos":ng,
          "geometrias_repetidas_no_ano":len(rep),"pct_geometrias_repetidas":100*len(rep)/ng if ng else np.nan,
          "registros_em_grupos_repetidos":int(rep.registros.sum()),"ocorrencias_adicionais_no_ano":total-ng,
          "geometrias_com_operacoes_distintas_no_ano":int(a.operacoes.gt(1).sum()),
          "geometrias_com_ordens_distintas_no_ano":int(a.ordens.gt(1).sum())})
    annual=pd.DataFrame(anual)
    temporal=g.groupby("categoria_temporal",sort=False).agg(geometrias=("registros","size"),registros=("registros","sum")).reset_index()
    temporal["ocorrencias_adicionais"]=temporal.registros-temporal.geometrias
    administrativo=g.groupby("categoria_administrativa",sort=False).agg(geometrias=("registros","size"),registros=("registros","sum")).reset_index()
    mesma_chave=d.groupby(["id_gleba2","ref_bacen","ordem"],sort=False).agg(
        registros=("linha_arquivo","size"),anos=("ano","nunique"),primeiro_ano=("ano","min"),ultimo_ano=("ano","max"),
        conteudos_distintos=("conteudo_sha256","nunique"),ids_gleba_distintos=("id_gleba","nunique")).reset_index()
    mesma_chave=mesma_chave.loc[mesma_chave.registros>1].copy()
    mesma_chave_ano=d.groupby(["id_gleba2","ano","ref_bacen","ordem"],sort=False).agg(
        registros=("linha_arquivo","size"),conteudos_distintos=("conteudo_sha256","nunique")).reset_index()
    mesma_chave_ano=mesma_chave_ano.loc[mesma_chave_ano.registros>1].copy()
    # Uma mesma operação em vários anos não equivale a várias contratações.
    operacoes_multianos=d.groupby("ref_bacen",sort=False).agg(anos=("ano","nunique"),registros=("linha_arquivo","size"),
        geometrias=("id_gleba2","nunique")).reset_index()
    operacoes_multianos=operacoes_multianos.loc[operacoes_multianos.anos>1]
    conjuntos={a:set(gy.loc[gy.ano.eq(a),"id_gleba2"]) for a in ANOS}
    matriz=pd.DataFrame([[len(conjuntos[a]&conjuntos[b]) for b in ANOS] for a in ANOS],index=ANOS,columns=ANOS)
    matriz.index.name="ano"
    assert int(temporal.registros.sum())==len(d) and int(temporal.geometrias.sum())==len(g)
    assert len(d)-len(g)==int(annual.ocorrencias_adicionais_no_ano.sum())+(len(gy)-len(g))
    return d,g,gy,rep_ano,annual,temporal,administrativo,mesma_chave,mesma_chave_ano,operacoes_multianos,matriz

(df,g,gy,rep_ano,anual,temporal,administrativo,mesma_chave,mesma_chave_ano,operacoes_multianos,matriz)=analisar_repeticoes(df)
for nome,t in {
    "perfil_por_id_gleba2.csv":g.reset_index(),"repeticoes_no_mesmo_ano.csv":rep_ano,
    "resumo_anual.csv":anual,"categorias_temporais.csv":temporal,"categorias_administrativas.csv":administrativo,
    "mesma_geometria_operacao_ordem.csv":mesma_chave,"mesma_geometria_operacao_ordem_ano.csv":mesma_chave_ano,
    "operacoes_em_mais_de_um_ano.csv":operacoes_multianos,"matriz_geometrias_compartilhadas_entre_anos.csv":matriz.reset_index(),
}.items():csv_saida(t,nome)
# Relatório de evidência em nível de linha, sem valores financeiros ou WKT extenso.
cols_evidencia=[x for x in ["linha_arquivo","ano","id_gleba","id_gleba2","ref_bacen","ordem","nu_indice","tipo_wkt","wkt_sha256"] if x in df]
repetidas=df.loc[df.id_gleba2.isin(g.index[g.registros>1]),cols_evidencia].sort_values(["id_gleba2","ano","ref_bacen","ordem","linha_arquivo"])
csv_saida(repetidas,"registros_com_repeticao_de_geometria.csv")
display(anual);display(temporal);display(administrativo)
print("IDs compartilhados entre anos (diagonal = distintos no ano):")
display(matriz)


,ano,registros,id_gleba2_distintos,geometrias_repetidas_no_ano,pct_geometrias_repetidas,registros_em_grupos_repetidos,ocorrencias_adicionais_no_ano,geometrias_com_operacoes_distintas_no_ano,geometrias_com_ordens_distintas_no_ano
0,2021,98209,85978,9971,11.597153,22202,12231,7560,9544
1,2022,84771,76464,6841,8.946694,15148,8307,4846,6409
2,2023,93911,82087,8636,10.520545,20460,11824,5218,8179
3,2024,105194,90794,10280,11.322334,24680,14400,5696,9773
4,2025,98594,88119,7583,8.605409,18058,10475,3937,7248


,categoria_temporal,geometrias,registros,ocorrencias_adicionais
0,Repetição entre anos sem repetição interna,16297,34448,18151
1,Repetição somente dentro de um ano,33626,78311,44685
2,Repetição entre anos e dentro de algum ano,7714,29982,22268
3,Uma ocorrência,337938,337938,0


,categoria_administrativa,geometrias,registros
0,Repetição com operações distintas,42803,107187
1,Uma ocorrência,337938,337938
2,Mesma operação com ordens distintas,12820,30894
3,Mesma operação e mesma ordem,2014,4660


IDs compartilhados entre anos (diagonal = distintos no ano):


,2021,2022,2023,2024,2025
ano,,,,,
2021,85978,12909,4332,1380,129
2022,12909,76464,5196,1574,134
2023,4332,5196,82087,4523,404
2024,1380,1574,4523,90794,1646
2025,129,134,404,1646,88119


## 7. Linhas idênticas e conteúdo repetido
Linhas iguais sem os identificadores artificiais são uma sinalização diferente da mera repetição de geometria. Mesmo assim, **não serão eliminadas**, nem rotuladas automaticamente como erro de concessão de crédito.

A segunda leitura abaixo confere, nos candidatos encontrados pelas assinaturas, a igualdade dos valores de todas as colunas consideradas. Mantém no relatório o total de grupos, as linhas envolvidas e as ocorrências adicionais. A comparação corresponde aos valores textuais interpretados do CSV; não é uma comparação de bytes de linhas ou de formatação das aspas.


In [ ]:
def resumir_duplicatas(d,coluna):
    t=d.groupby(coluna,sort=False).agg(registros=("linha_arquivo","size"),primeira_linha=("linha_arquivo","min")).reset_index()
    t=t.loc[t.registros>1].copy();t["ocorrencias_adicionais"]=t.registros-1
    return t

dup_integral=resumir_duplicatas(df,"linha_sha256")
dup_conteudo=resumir_duplicatas(df,"conteudo_sha256")

def conferir_igualdade_candidatos():
    candidatos={"linha_sha256":set(dup_integral.linha_sha256),"conteudo_sha256":set(dup_conteudo.conteudo_sha256)}
    if not any(candidatos.values()):return "sem candidatos; segunda leitura dispensada"
    refs={k:{} for k in candidatos}
    linhas_candidatas=df.loc[df.linha_sha256.isin(candidatos["linha_sha256"])|df.conteudo_sha256.isin(candidatos["conteudo_sha256"]),
                             ["linha_arquivo","linha_sha256","conteudo_sha256"]].set_index("linha_arquivo")
    pos=0;verificadas=0
    for bloco in pd.read_csv(CAMINHO,sep=SEPARADOR,encoding=CODIFICACAO,dtype=str,keep_default_na=False,chunksize=TAMANHO_BLOCO,
                             on_bad_lines="error",skip_blank_lines=False):
        bloco=bloco.fillna("").reset_index(drop=True)
        selecionadas=linhas_candidatas.loc[(linhas_candidatas.index>pos)&(linhas_candidatas.index<=pos+len(bloco))]
        for linha,assinaturas in selecionadas.iterrows():
            registro=bloco.iloc[int(linha)-pos-1]
            for k,colunas in [("linha_sha256",CABECALHO),("conteudo_sha256",COLUNAS_SEM_IDS)]:
                sig=assinaturas[k]
                if sig not in candidatos[k]:continue
                valores=tuple(registro[colunas])
                if assinatura_valores(valores)!=sig:raise RuntimeError("A entrada mudou entre leituras; execute novamente em uma base estável.")
                if sig in refs[k] and refs[k][sig]!=valores:raise RuntimeError("Colisão de assinatura: a igualdade não foi confirmada.")
                refs[k][sig]=valores
            verificadas+=1
        pos+=len(bloco)
    if verificadas!=len(linhas_candidatas):raise RuntimeError("Nem todos os candidatos foram conferidos.")
    return f"igualdade confirmada em {verificadas} linhas candidatas"

STATUS_DUPLICATAS=conferir_igualdade_candidatos()
csv_saida(dup_integral,"grupos_linhas_integralmente_iguais.csv")
csv_saida(dup_conteudo,"grupos_conteudo_igual_sem_ids_artificiais.csv")
linhas_dup=df.loc[df.conteudo_sha256.isin(set(dup_conteudo.conteudo_sha256)),cols_evidencia+["conteudo_sha256"]]
csv_saida(linhas_dup,"linhas_conteudo_igual_sem_ids_artificiais.csv")
resumo_dup=pd.DataFrame([
    {"criterio":nome,"grupos":len(t),"linhas_envolvidas":int(t.registros.sum()),"ocorrencias_adicionais":int(t.ocorrencias_adicionais.sum())}
    for nome,t in [("Todas as colunas",dup_integral),("Sem ID_GLEBA e ID_GLEBA2",dup_conteudo)]
])
csv_saida(resumo_dup,"resumo_duplicacoes_conteudo.csv");display(resumo_dup)
print(STATUS_DUPLICATAS)


,criterio,grupos,linhas_envolvidas,ocorrencias_adicionais
0,Todas as colunas,0,0,0
1,Sem ID_GLEBA e ID_GLEBA2,0,0,0


sem candidatos; segunda leitura dispensada


## 8. Beneficiário e indicadores de síntese
Sem uma coluna que identifique o beneficiário, a conclusão possível é “operações diferentes vinculadas ao mesmo WKT”. Não é possível afirmar que o mesmo produtor pediu mais de um financiamento.

Se a coluna foi explicitamente configurada, os resultados também mostram sua cobertura. Um único identificador observado com linhas sem identificação não comprova que todos os registros sejam do mesmo beneficiário.


In [ ]:
perfil_benef=None
if BENEFICIARIO:
    perfil_benef=df.groupby("id_gleba2",sort=False).agg(
        registros=("linha_arquivo","size"),registros_com_beneficiario=("beneficiario_hash","count"),
        beneficiarios_distintos=("beneficiario_hash","nunique")).reset_index()
    perfil_benef["cobertura_completa"]=perfil_benef.registros.eq(perfil_benef.registros_com_beneficiario)
    csv_saida(perfil_benef,"beneficiarios_por_id_gleba2.csv")
    print("Cobertura da coluna de beneficiário:",f"{100*df.beneficiario_hash.notna().mean():.2f}%")
    display(perfil_benef.head(10))

metricas={
    "registros":len(df),"id_gleba2_distintos":len(g),"geometrias_com_mais_de_um_registro":int(g.registros.gt(1).sum()),
    "registros_em_geometrias_repetidas":int(g.loc[g.registros.gt(1),"registros"].sum()),
    "ocorrencias_adicionais_totais":len(df)-len(g),
    "geometrias_repetidas_em_algum_ano":int(g.repete_dentro_de_algum_ano.sum()),
    "geometrias_em_mais_de_um_ano":int(g.anos.gt(1).sum()),
    "geometrias_com_operacoes_distintas":int(g.operacoes.gt(1).sum()),
    "geometrias_com_ordens_distintas":int(g.ordens.gt(1).sum()),
    "geometrias_com_operacoes_distintas_em_algum_mesmo_ano":gy.loc[gy.operacoes.gt(1),"id_gleba2"].nunique(),
    "grupos_geometria_operacao_ordem_repetidos":len(mesma_chave),
    "grupos_geometria_operacao_ordem_ano_repetidos":len(mesma_chave_ano),
    "operacoes_presentes_em_mais_de_um_ano":len(operacoes_multianos),
    "ocorrencias_adicionais_dentro_dos_anos":int(anual.ocorrencias_adicionais_no_ano.sum()),
    "presencas_adicionais_em_outros_anos":len(gy)-len(g),
    "linhas_integralmente_duplicadas_adicionais":int(dup_integral.ocorrencias_adicionais.sum()),
    "linhas_conteudo_igual_sem_ids_adicionais":int(dup_conteudo.ocorrencias_adicionais.sum()),
}
indicadores=pd.DataFrame([{"indicador":k,"quantidade":int(v)} for k,v in metricas.items()])
csv_saida(indicadores,"indicadores_gerais.csv")
tipos=df.groupby("tipo_wkt",dropna=False).agg(registros=("linha_arquivo","size"),ids_distintos=("id_gleba2","nunique")).reset_index()
csv_saida(tipos,"tipos_geometricos_por_texto_wkt.csv")
display(indicadores)
print("Identidade de conferência: ocorrências adicionais totais = adicionais dentro dos anos + presenças adicionais em outros anos.")
print("Esta identidade de contagem não atribui causas financeiras às repetições.")


,indicador,quantidade
0,registros,480679
1,id_gleba2_distintos,395575
2,geometrias_com_mais_de_um_registro,57637
3,registros_em_geometrias_repetidas,142741
4,ocorrencias_adicionais_totais,85104
5,geometrias_repetidas_em_algum_ano,41340
6,geometrias_em_mais_de_um_ano,24011
7,geometrias_com_operacoes_distintas,42803
8,geometrias_com_ordens_distintas,55623
9,geometrias_com_operacoes_distintas_em_algum_me...,25647


Identidade de conferência: ocorrências adicionais totais = adicionais dentro dos anos + presenças adicionais em outros anos.
Esta identidade de contagem não atribui causas financeiras às repetições.


## 9. Respostas para o orientador e tabelas para o Word
O arquivo de texto adapta as respostas às contagens efetivamente calculadas. As tabelas HTML são copiáveis para o Word. Os números **não são preenchidos a partir dos resultados anteriores**.

Os totais encontrados aqui não comprovam que os cruzamentos espaciais dos demais notebooks tenham sido executados uma única vez por geometria. O `id_gleba2` permite esse uso, mas verificar sua implementação exige examinar os notebooks de interseção.


In [ ]:
def br(n):return f"{int(n):,}".replace(",",".")
def pct(n,d):return "não calculável" if not d else f"{100*n/d:.4f}%".replace(".",",")
m=metricas
linhas=["5.3.1 — Evidências sobre repetição das representações geométricas", "",
 f"Foram examinados {br(m['registros'])} registros de 2021 a 2025, com {br(m['id_gleba2_distintos'])} valores distintos de id_gleba2. O ano utilizado é {MAPA['ano']}, correspondente ao ano de origem do arquivo territorial.",
 f"A auditoria identificou {br(m['geometrias_com_mais_de_um_registro'])} representações WKT presentes em mais de um registro ({pct(m['geometrias_com_mais_de_um_registro'],m['id_gleba2_distintos'])} dos IDs distintos). Esses grupos reúnem {br(m['registros_em_geometrias_repetidas'])} registros, com {br(m['ocorrencias_adicionais_totais'])} ocorrências adicionais à primeira de cada representação.", "",
 "1. Existem repetições no mesmo ano?",
 (f"Sim. {br(m['geometrias_repetidas_em_algum_ano'])} IDs se repetem dentro de pelo menos um ano de origem." if m['geometrias_repetidas_em_algum_ano'] else "Não foram identificados IDs repetidos dentro de um mesmo ano de origem nos arquivos analisados.")]
for _,r in anual.iterrows():
    linhas.append(f"{int(r.ano)}: {br(r.registros)} registros; {br(r.id_gleba2_distintos)} IDs distintos; {br(r.geometrias_repetidas_no_ano)} IDs repetidos no ano ({pct(r.geometrias_repetidas_no_ano,r.id_gleba2_distintos)} dos IDs desse ano); {br(r.ocorrencias_adicionais_no_ano)} ocorrências adicionais.")
linhas += ["", "2. A representação reaparece em anos diferentes?",
 f"Foram identificados {br(m['geometrias_em_mais_de_um_ano'])} IDs presentes em mais de um ano de origem. Esse número pode se sobrepor ao de IDs repetidos dentro de algum ano; consulte as categorias temporais exclusivas para não somá-los indevidamente.",
 "", "3. São financiamentos diferentes para a mesma gleba?",
 f"Em {br(m['geometrias_com_operacoes_distintas'])} IDs há associação a mais de uma referência REF_BACEN. Em {br(m['geometrias_com_operacoes_distintas_em_algum_mesmo_ano'])} IDs, essa associação a operações distintas ocorre dentro de pelo menos um mesmo ano de origem. Há ainda {br(m['geometrias_com_ordens_distintas'])} IDs associados a mais de um par REF_BACEN + NU_ORDEM_GLEBA.",
 "Esses resultados descrevem operações e ordens registradas vinculadas à mesma representação territorial. Não comprovam, isoladamente, quem solicitou o crédito, a aprovação, o desembolso ou a identidade do produtor.",
 "", "4. É o mesmo produtor?",
 ("A coluna de beneficiário não foi informada. Não é possível responder a essa pergunta com as chaves territoriais e administrativas analisadas. Categoria do emitente, porte e programa não identificam o beneficiário." if BENEFICIARIO is None else
 f"Foi examinada a coluna {BENEFICIARIO}, com identificação preenchida em {br(df.beneficiario_hash.notna().sum())} de {br(len(df))} registros. O arquivo beneficiarios_por_id_gleba2.csv separa quantidade de identificadores e cobertura. A validade do campo como identificação do beneficiário depende de sua documentação; nomes ou códigos parciais podem não individualizar pessoas."),
 "", "5. A repetição é uma inconsistência?",
 f"A repetição geométrica não caracteriza, por si só, inconsistência. Foram encontrados {br(m['grupos_geometria_operacao_ordem_repetidos'])} grupos repetidos com a mesma geometria, operação e ordem; ao acrescentar o ano à chave, {br(m['grupos_geometria_operacao_ordem_ano_repetidos'])} grupos permanecem repetidos.",
 f"Na comparação de todas as colunas, foram encontradas {br(m['linhas_integralmente_duplicadas_adicionais'])} linhas adicionais idênticas. Desconsiderando somente ID_GLEBA e ID_GLEBA2, foram encontradas {br(m['linhas_conteudo_igual_sem_ids_adicionais'])} linhas adicionais de conteúdo idêntico. A igualdade dos candidatos foi conferida nos valores lidos.",
 "Esses casos exigem exame de origem, versão dos arquivos e atributos administrativos antes de serem classificados como erro. Nenhuma linha foi eliminada e nenhuma repetição foi interpretada automaticamente como financiamento irregular.",
 "", "6. Como o identificador foi construído?",
 "O código Base05 consultado usa SHA-256 do WKT exato e atribui ao grupo o menor ID_GLEBA. Não realiza normalização geométrica. WKT diferentes podem representar uma mesma área e permanecer em grupos distintos. Assim, a redação deve referir-se a repetições da mesma representação WKT, e não a todas as áreas fisicamente equivalentes.",
 f"A correspondência ID_GLEBA2 ↔ assinatura WKT foi conferida sem conflitos. Regra do menor ID: {verificacao_menor_id}; linhas divergentes identificadas: {br(len(divergencias)) if verificacao_menor_id=='verificada' else 'não avaliadas'}. Foram encontrados {br(ids_orig_repetidos.id_gleba.nunique())} valores de ID_GLEBA presentes em mais de uma linha.",
 "", "Limites adicionais",
 "O ano de origem não é automaticamente o ano da contratação. A igualdade textual não é um teste de igualdade topológica nem de propriedade, e a auditoria não verifica a reutilização dos cruzamentos espaciais nos demais notebooks."]
texto="\n".join(linhas)
(SAIDA/"respostas_ao_orientador_5_3_1.txt").write_text(texto,encoding="utf-8")
print(texto)

def tabela_html(titulo,tabela,nome,nota):
    import html
    doc=("<!doctype html><html lang='pt-BR'><meta charset='utf-8'><style>body{font:14px Arial;margin:28px}"
         "table{border-collapse:collapse}th,td{padding:8px;border:1px solid #d9d9d9;text-align:right}th{background:#e8edde}"
         "p{line-height:1.5}</style><h2>"+html.escape(titulo)+"</h2>"+tabela.to_html(index=False,escape=True,border=0)+"<p>"+html.escape(nota)+"</p></html>")
    (SAIDA/nome).write_text(doc,encoding="utf-8")
tab=anual[["ano","registros","id_gleba2_distintos","geometrias_repetidas_no_ano","pct_geometrias_repetidas","ocorrencias_adicionais_no_ano","geometrias_com_operacoes_distintas_no_ano"]].copy()
for col in tab:
    if col=="pct_geometrias_repetidas":tab[col]=tab[col].map(lambda v:"—" if pd.isna(v) else f"{v:.4f}".replace(".",","))
    elif col!="ano":tab[col]=tab[col].map(br)
tab.columns=["Ano de origem","Registros","IDs distintos","IDs repetidos no ano","IDs repetidos (%)","Ocorrências adicionais","IDs com operações distintas"]
tabela_html("Repetições por ano de origem",tab,"tabela_word_repeticoes_anuais.html","Percentual = IDs repetidos no ano / IDs distintos no mesmo ano × 100. IDs de anos diferentes podem se repetir; as colunas não fornecem totais distintos do período por simples soma.")
tabela_html("Padrões temporais de repetição",temporal,"tabela_word_categorias_temporais.html","Categorias exclusivas. As ocorrências adicionais equivalem aos registros menos os IDs distintos de cada categoria.")
tabela_html("Relação entre geometrias e identificadores administrativos",administrativo,"tabela_word_categorias_administrativas.html","Categorias exclusivas definidas pelo perfil completo de cada ID no período. Operação = REF_BACEN; ordem = REF_BACEN + NU_ORDEM_GLEBA.")
display(HTML(tab.to_html(index=False)))


5.3.1 — Evidências sobre repetição das representações geométricas

Foram examinados 480.679 registros de 2021 a 2025, com 395.575 valores distintos de id_gleba2. O ano utilizado é ANO_BASE, correspondente ao ano de origem do arquivo territorial.
A auditoria identificou 57.637 representações WKT presentes em mais de um registro (14,5704% dos IDs distintos). Esses grupos reúnem 142.741 registros, com 85.104 ocorrências adicionais à primeira de cada representação.

1. Existem repetições no mesmo ano?
Sim. 41.340 IDs se repetem dentro de pelo menos um ano de origem.
2021: 98.209 registros; 85.978 IDs distintos; 9.971 IDs repetidos no ano (11,5972% dos IDs desse ano); 12.231 ocorrências adicionais.
2022: 84.771 registros; 76.464 IDs distintos; 6.841 IDs repetidos no ano (8,9467% dos IDs desse ano); 8.307 ocorrências adicionais.
2023: 93.911 registros; 82.087 IDs distintos; 8.636 IDs repetidos no ano (10,5205% dos IDs desse ano); 11.824 ocorrências adicionais.
2024: 105.194 registros; 90.794

Ano de origem,Registros,IDs distintos,IDs repetidos no ano,IDs repetidos (%),Ocorrências adicionais,IDs com operações distintas
2021,98.209,85.978,9.971,"11,5972",12.231,7.560
2022,84.771,76.464,6.841,"8,9467",8.307,4.846
2023,93.911,82.087,8.636,"10,5205",11.824,5.218
2024,105.194,90.794,10.280,"11,3223",14.400,5.696
2025,98.594,88.119,7.583,"8,6054",10.475,3.937


## 10. Salvar metadados e baixar o ZIP
Os resultados são salvos em uma pasta própria no Drive, sem substituir a Base04. O ZIP contém todos os CSVs, tabelas HTML e o texto de respostas. Compartilhe esses resultados para preparar a redação definitiva do subcapítulo.

In [ ]:
metadados={"gerado_em_utc":datetime.now(timezone.utc).isoformat(),"arquivo":str(CAMINHO),"drive_id":DRIVE_ID_BASE,
    "metadados_entrada":META_ENTRADA,"colunas":MAPA,"coluna_indice":INDICE,"coluna_beneficiario":BENEFICIARIO,
    "colunas_comparadas_sem_ids":COLUNAS_SEM_IDS,"anos":ANOS,"ano_referencia":"Ano do arquivo territorial",
    "unidade":"Registro territorial; repetições de representação WKT exata por ID_GLEBA2",
    "regra_base05":"SHA256 do WKT exato; identificador igual ao menor ID_GLEBA do grupo",
    "verificacao_duplicatas":STATUS_DUPLICATAS,"verificacao_menor_id":verificacao_menor_id,
    "deduplicacao_da_base":False,"base_modificada":False,"resultados_calculados":{k:int(v) for k,v in metricas.items()},
    "versoes":{"python":platform.python_version(),"pandas":pd.__version__,"numpy":np.__version__},
    "nota":"Validade topológica não foi reavaliada; o tipo foi extraído do WKT. Beneficiário ausente não permite inferir identidade do produtor."}
(SAIDA/"metadados.json").write_text(json.dumps(metadados,ensure_ascii=False,indent=2),encoding="utf-8")
zip_final=SAIDA/"Auditoria_Repeticoes_ID_GLEBA2_Resultados.zip"
with zipfile.ZipFile(zip_final,"w",zipfile.ZIP_DEFLATED) as z:
    for p in sorted(SAIDA.iterdir()):
        if p.is_file() and p!=zip_final:z.write(p,arcname=p.name)
print("RESULTADOS:",SAIDA)
print("ZIP:",zip_final)
if BAIXAR_ZIP:
    from google.colab import files
    files.download(str(zip_final))


RESULTADOS: /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/auditoria_repeticoes_5_3_1/20260917_224431_688524
ZIP: /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/auditoria_repeticoes_5_3_1/20260917_224431_688524/Auditoria_Repeticoes_ID_GLEBA2_Resultados.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>